# Predicting Airline Satisfaction · EDA + LightGBM baseline

**Kaggle Playground Series S6E10** · [competition](https://www.kaggle.com/competitions/playground-series-s6e10) · [GitHub repo](https://github.com/aditbytes/kaggle-s6e10-airline-satisfaction)

**The task.** For each passenger in `test.csv`, predict the probability that they were **satisfied** with their flight. We get their trip details (age, travel class, distance, delays) and 13 service ratings from 0 to 5.

**The metric: ROC AUC.** AUC scores how well predictions *rank* passengers: it is the chance that a random satisfied passenger gets a higher score than a random unsatisfied one. 0.5 = coin flip, 1.0 = perfect. Only the **order** of predictions matters, so they don't need to be calibrated probabilities.

**Contents:** 1 Setup · 2 Data overview · 3 EDA · 4 Cross-validation · 5 LightGBM · 6 Feature importance · 7 Submission · 8 What I tried. Runs top to bottom on a CPU.

## 1 · Setup

Imports, the data folder, and the constants used everywhere. The notebook finds the data both on Kaggle (`/kaggle/input/...`) and locally inside the repo (`../data`). A fixed random seed makes every run reproducible.

In [ ]:
import json
import os
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

T0 = time.time()
ON_KAGGLE = Path("/kaggle/input").exists()


def find_data_dir() -> Path:
    candidates = [Path("/kaggle/input/playground-series-s6e10"),
                  Path("/kaggle/input/competitions/playground-series-s6e10"),
                  Path("../data"), Path("data")]
    for c in candidates:
        if (c / "train.csv").exists():
            return c
    if ON_KAGGLE:  # fall back to any attached folder that looks like this competition
        for p in Path("/kaggle/input").glob("**/train.csv"):
            if (p.parent / "sample_submission.csv").exists():
                return p.parent
    raise FileNotFoundError("train.csv not found: attach the competition data (Kaggle) or download it into ../data")


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(".")

ID, TARGET = "id", "satisfaction"
SEED, N_FOLDS = 42, 5
CATEGORICAL = ["Gender", "Customer Type", "Type of Travel", "Class"]
SERVICE_RATINGS = [
    "Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
    "Gate location", "Food and drink", "Online boarding", "Seat comfort", "Inflight entertainment",
    "On-board service", "Leg room service", "Baggage handling", "Checkin service", "Cleanliness",
]
DELAYS = ["Departure Delay in Minutes", "Arrival Delay in Minutes"]
NUMERIC = ["Age", "Flight Distance", *DELAYS]
FEATURES = NUMERIC + SERVICE_RATINGS + CATEGORICAL

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"data: {DATA_DIR} | on Kaggle: {ON_KAGGLE} | CPUs: {os.cpu_count()} | lightgbm {lgb.__version__}")

## 2 · Data overview

`train.csv` includes the answer (`satisfaction`); `test.csv` doesn't, so those are the rows we predict. The four text columns become pandas `category` columns, which LightGBM can split on directly (no one-hot encoding needed), and the `True`/`False` target becomes 1/0.

In [ ]:
def load(name: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / name)
    for col in CATEGORICAL:
        if col in df:
            df[col] = df[col].astype("category")
    return df.set_index(ID) if ID in df else df


train = load("train.csv")
test = load("test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
train[TARGET] = train[TARGET].astype(str).str.lower().eq("true").astype("int8")
# give train and test the same category lists so codes mean the same thing in both
for col in CATEGORICAL:
    cats = sorted(set(train[col].cat.categories) | set(test[col].cat.categories))
    train[col] = train[col].cat.set_categories(cats)
    test[col] = test[col].cat.set_categories(cats)

print(f"train {train.shape} | test {test.shape} | sample_submission {sample.shape}")
train.head(3)

In [ ]:
print(f"Satisfied: {train[TARGET].mean():.2%} of {len(train):,} passengers")
missing = pd.DataFrame({"train": train.isna().sum(), "test": test.isna().sum()}).fillna(0).astype(int)
missing[(missing["train"] > 0) | (missing["test"] > 0)]

**Takeaways:** about 44% of passengers are satisfied, so the classes are fairly balanced. The only missing values are a few hundred `Arrival Delay in Minutes`. LightGBM handles missing values natively, so we leave them as they are.

## 3 · Exploratory data analysis

### 3.1 Target balance and segments

How many passengers were satisfied, and how does that change across **travel class**, **type of travel** and **customer type**? Big differences between groups mean the column is useful for the model.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
counts = train[TARGET].map({0: "not satisfied", 1: "satisfied"}).value_counts()
axes[0].bar(counts.index, counts.values, color=["#b0b7c3", "#2a78b5"])
axes[0].set_title("Target balance")
axes[0].set_ylabel("passengers")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f"{v / len(train):.1%}", ha="center", va="bottom")

for ax, col in zip(axes[1:], ["Class", "Type of Travel", "Customer Type"]):
    rate = train.groupby(col, observed=True)[TARGET].mean().sort_values()
    ax.barh(rate.index.astype(str), rate.values, color="#2a78b5")
    ax.axvline(train[TARGET].mean(), color="#555", ls="--", lw=1)
    ax.set_xlim(0, 1)
    ax.set_title(f"Satisfied rate by {col}")
    for y, v in enumerate(rate.values):
        ax.text(v + 0.02, y, f"{v:.0%}", va="center")
plt.tight_layout()
plt.show()

**Takeaways:** the segments split satisfaction sharply. **Business class** passengers are ~73% satisfied vs ~17% in Eco. **Personal travel** is only ~10% satisfied, and **disloyal** customers ~20% (the dashed line is the overall 44%).

### 3.2 Which service ratings differ most?

For each 0–5 rating, the average among satisfied passengers minus the average among unsatisfied ones. A bigger gap means satisfied passengers rated that service much higher.

In [ ]:
means = train.groupby(TARGET)[SERVICE_RATINGS].mean().T
gap = (means[1] - means[0]).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(gap.index, gap.values, color=np.where(gap.values >= 0, "#2a78b5", "#c0504d"))
ax.axvline(0, color="#555", lw=1)
ax.set_title("Rating gap: mean if satisfied minus mean if not (0-5 scale)")
for y, v in enumerate(gap.values):
    ax.text(v + (0.02 if v >= 0 else -0.02), y, f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right")
plt.tight_layout()
plt.show()

### 3.3 How predictive is each feature on its own?

The AUC you'd get using one column alone as the prediction (categories are replaced by their satisfied rate). 0.5 means no signal. This is a quick way to spot the strongest features before modelling.

In [ ]:
y = train[TARGET].to_numpy()
single_auc = {}
for col in FEATURES:
    x = train[col]
    if col in CATEGORICAL:
        x = x.map(train.groupby(col, observed=True)[TARGET].mean()).astype(float)
    auc = roc_auc_score(y, x.fillna(x.median()))
    single_auc[col] = max(auc, 1 - auc)  # direction doesn't matter for "how much signal"
single_auc = pd.Series(single_auc).sort_values()

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(single_auc.index, single_auc.values - 0.5, left=0.5, color="#2a78b5")
ax.set_xlim(0.5, 0.9)
ax.set_title("Single-feature ROC AUC (0.5 = no signal)")
for y_pos, v in enumerate(single_auc.values):
    ax.text(v + 0.003, y_pos, f"{v:.3f}", va="center")
plt.tight_layout()
plt.show()

**Takeaways:** `Online boarding` alone already reaches AUC ≈ 0.84, followed by `Class`, `Inflight entertainment` and `Seat comfort`. Delays, `Gender` and `Gate location` are close to 0.5 on their own, though trees can still use them in combination with other features.

## 4 · Cross-validation

Kaggle hides the test answers, so we hold out part of `train` to measure our model.

**Stratified 5-fold:** split train into 5 parts, each with the same share of satisfied passengers. Train on 4 parts, validate on the 5th, and repeat 5 times. Every row gets exactly one **out-of-fold (OOF)** prediction, and the AUC over all OOF predictions is our **CV score**.

These are the same folds (seed 42) as `src/cv.py` in the repo, so scores here are comparable with every experiment there.

In [ ]:
def make_folds(y: pd.Series, n_folds: int = N_FOLDS, seed: int = SEED) -> np.ndarray:
    """Fold number (0..n_folds-1) for every row, stratified on the target."""
    folds = np.full(len(y), -1, dtype=np.int8)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold, (_, valid_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
        folds[valid_idx] = fold
    return folds


folds = make_folds(train[TARGET])
pd.DataFrame({"rows": np.bincount(folds),
              "satisfied rate": [train[TARGET].to_numpy()[folds == k].mean() for k in range(N_FOLDS)]},
             index=pd.Index(range(N_FOLDS), name="fold")).round(4)